# Actividad 4 — Punto 1: Comparación de modelos preentrenados
## ConectaSegura: análisis de sentimiento de reportes sobre llamadas sospechosas

Este prototipo compara tres modelos de Hugging Face para identificar el tono de los reportes en español (NEG, NEU o POS). El resultado apoya la priorización de reportes; no bloquea llamadas ni sustituye la revisión humana.

### 1. Configuración de GPU
En Google Colab selecciona: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.

In [ ]:
!nvidia-smi
import torch
DEVICE = 0 if torch.cuda.is_available() else -1
print('Dispositivo:', 'GPU' if DEVICE == 0 else 'CPU')

### 2. Dependencias y autenticación opcional
Los modelos seleccionados son públicos; el token solo se usa para cumplir la conexión segura al Hub o si después se desea publicar el prototipo.

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub scikit-learn pandas matplotlib seaborn
from huggingface_hub import login
# Opcional: activa la siguiente línea y pega un token con permiso Read cuando Colab lo solicite.
# login()

### 3. Dataset de evaluación
Son ejemplos etiquetados manualmente que representan reportes de clientes. Para el prototipo se usan 30 observaciones balanceadas; en producción deben reemplazarse por datos anonimizados, representativos y revisados.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time

datos = [
 ('Me llamaron para pedirme mi contraseña y datos de mi tarjeta.', 'NEG'),
 ('Recibí varias llamadas con amenazas para depositar dinero.', 'NEG'),
 ('El número insistió en que compartiera el código de verificación.', 'NEG'),
 ('La persona fingió ser del banco y solicitó información privada.', 'NEG'),
 ('Me ofrecieron un premio, pero querían que hiciera una transferencia.', 'NEG'),
 ('La llamada fue agresiva y me presionaron para pagar de inmediato.', 'NEG'),
 ('Reporto una llamada sospechosa que intentó obtener mis datos.', 'NEG'),
 ('Usaron mi nombre y dijeron que mi cuenta estaba comprometida.', 'NEG'),
 ('Me marcaron desde números distintos con el mismo engaño.', 'NEG'),
 ('La llamada parecía fraude porque pidieron acceso a mi aplicación.', 'NEG'),
 ('Recibí una llamada desconocida y deseo verificar el número.', 'NEU'),
 ('El teléfono marcó dos veces, pero no dejaron mensaje.', 'NEU'),
 ('Quiero saber cómo reportar un número que no reconozco.', 'NEU'),
 ('La llamada se cortó antes de que pudiera contestar.', 'NEU'),
 ('Tengo una llamada perdida de un número sin identificar.', 'NEU'),
 ('¿Pueden revisar si este número tiene reportes?', 'NEU'),
 ('Me contactaron para una encuesta y no di información.', 'NEU'),
 ('Deseo consultar las opciones de seguridad para llamadas.', 'NEU'),
 ('El número no estaba en mis contactos y quiero información.', 'NEU'),
 ('Solicito orientación sobre una llamada recibida hoy.', 'NEU'),
 ('Gracias por avisarme que el número tenía reportes de fraude.', 'POS'),
 ('La alerta de ConectaSegura me ayudó a evitar una estafa.', 'POS'),
 ('El proceso para reportar la llamada fue claro y rápido.', 'POS'),
 ('Me atendieron bien y resolvieron mi reporte.', 'POS'),
 ('La aplicación identificó el riesgo antes de que respondiera.', 'POS'),
 ('Agradezco la protección y las recomendaciones recibidas.', 'POS'),
 ('La notificación fue útil para reconocer el intento de engaño.', 'POS'),
 ('El canal de soporte respondió correctamente a mi consulta.', 'POS'),
 ('Me siento más seguro con las alertas de llamadas sospechosas.', 'POS'),
 ('La información de prevención fue muy útil.', 'POS')
]
df = pd.DataFrame(datos, columns=['texto', 'etiqueta_real'])
display(df.head())
print(df['etiqueta_real'].value_counts())

### 4. Modelos seleccionados
Se comparan dos modelos especializados en español y uno multilingüe. La evaluación se hace con exactamente el mismo conjunto de mensajes.

In [ ]:
from transformers import pipeline

MODELOS = {
 'RoBERTuito': 'pysentimiento/robertuito-sentiment-analysis',
 'XLM-RoBERTa': 'cardiffnlp/twitter-xlm-roberta-base-sentiment',
 'DistilBERT multilingüe': 'lxyuan/distilbert-base-multilingual-cased-sentiments-student'
}

def normalizar_etiqueta(etiqueta):
    e = etiqueta.upper().replace('_', '').replace('-', '')
    mapa = {
        'POS': 'POS', 'POSITIVE': 'POS', 'LABEL2': 'POS', '5STARS': 'POS', '4STARS': 'POS',
        'NEG': 'NEG', 'NEGATIVE': 'NEG', 'LABEL0': 'NEG', '1STAR': 'NEG', '2STARS': 'NEG',
        'NEU': 'NEU', 'NEUTRAL': 'NEU', 'LABEL1': 'NEU', '3STARS': 'NEU'
    }
    return mapa.get(e, 'NEU')

resultados = []
predicciones_por_modelo = {}
textos = df['texto'].tolist()
reales = df['etiqueta_real'].tolist()

for nombre, modelo_id in MODELOS.items():
    print(f'\nEvaluando {nombre}...')
    clasificador = pipeline('sentiment-analysis', model=modelo_id, tokenizer=modelo_id, device=DEVICE, truncation=True)
    if DEVICE == 0: torch.cuda.synchronize()
    inicio = time.perf_counter()
    salidas = clasificador(textos, batch_size=8)
    if DEVICE == 0: torch.cuda.synchronize()
    latencia_total = time.perf_counter() - inicio
    pred = [normalizar_etiqueta(s['label']) for s in salidas]
    precision, recall, f1, _ = precision_recall_fscore_support(reales, pred, average='macro', zero_division=0)
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(reales, pred),
        'Precision_macro': precision,
        'Recall_macro': recall,
        'F1_macro': f1,
        'Latencia_total_s': latencia_total,
        'Latencia_promedio_ms': latencia_total / len(textos) * 1000
    })
    predicciones_por_modelo[nombre] = pred

tabla_resultados = pd.DataFrame(resultados).sort_values(['F1_macro', 'Latencia_promedio_ms'], ascending=[False, True])
display(tabla_resultados.style.format({
    'Accuracy': '{:.3f}', 'Precision_macro': '{:.3f}', 'Recall_macro': '{:.3f}',
    'F1_macro': '{:.3f}', 'Latencia_total_s': '{:.3f}', 'Latencia_promedio_ms': '{:.2f}'
}))

In [ ]:
# Tabla de predicciones, matrices de confusión y gráfica de comparación
comparacion = df.copy()
for nombre, pred in predicciones_por_modelo.items():
    comparacion[nombre] = pred
display(comparacion)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (nombre, pred) in zip(axes, predicciones_por_modelo.items()):
    cm = confusion_matrix(reales, pred, labels=['NEG', 'NEU', 'POS'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NEG','NEU','POS'], yticklabels=['NEG','NEU','POS'], ax=ax)
    ax.set_title(nombre)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Etiqueta real')
plt.tight_layout()
plt.show()

tabla_grafica = tabla_resultados.set_index('Modelo')
tabla_grafica[['F1_macro', 'Latencia_promedio_ms']].plot(kind='bar', secondary_y='Latencia_promedio_ms', figsize=(10,5), rot=0)
plt.title('Trade-off: F1-score frente a latencia')
plt.ylabel('F1-score')
plt.tight_layout()
plt.show()

### 5. Conclusión parcial
El modelo recomendado debe combinar un F1 macro alto con una latencia compatible con la operación. Para ConectaSegura se propone aceptar un modelo solo si alcanza **F1 macro ≥ 0.80**, **recall NEG ≥ 0.80** y una latencia promedio menor de **500 ms** por reporte en el entorno de prueba. La decisión final se registrará después de ejecutar el cuaderno y revisar errores por clase.

**Ajustes mínimos aplicados:** uso de GPU cuando está disponible, inferencia por lotes, truncamiento de textos y normalización de etiquetas. No se realiza fine-tuning en esta etapa.